# Figure G05: agreement of GTDB candidates with an independent gene caller (Prodigal)

**Status:** draft, Gadi. The ID is provisional. This notebook replaces the bare `Validation/gtdb/figures/prodigal_validation.png` (genome_entropy #116).

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figG05_prodigal_by_arm.tsv.gz` | `Validation/gtdb/matched_comparator/{bac,arc}_v7_matched.tsv`, `prodigal_*` rows | `41_comparator_summary.py` on `24_prodigal_overlap.py` output, v7 matched comparator |
| `data/figG05_prodigal_frame_class.tsv.gz` | Gadi `missed_genes/full_bac/prodigal_bac4.coincidence.tsv.gz` (sha256 `1e3f395e…`), `missed_genes/pilot_arc/prodigal_arc4.coincidence.tsv.gz` (`1a31fee8…`) | `24_prodigal_overlap.py`, original `shadow_hi` arm (before frame filtering and before rematching) |

- **Source:** `linsalrob/genome_entropy` `main` at `97a792f`.
- **Reduction:** `manuscript/prep/gtdb_prepare_data.py --only G05`. Panel b counts (n, n_coincides) per frame class; panel a copies the 41 rates verbatim.
- **Definitions:**
  - **Coincides** means the ORF shares a stop codon and strand with a Prodigal call on the GTDB genome (`gtdb_prodigal_coords`). Prodigal uses neither GenBank annotation nor structure databases.
  - **Prodigal share** = (candidate − background) / (annotated − background). The report's rule takes the larger of the two backgrounds, which is now the matched clean comparator in both domains.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}


In [ ]:
m = matched(DATA / "figG05_prodigal_by_arm.tsv.gz")
fc = pd.read_csv(DATA / "figG05_prodigal_frame_class.tsv.gz", sep="\t")
DOMS = ["bacteria", "archaea"]

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), gridspec_kw={"wspace": 0.32})

ax = axes[0]
arms = ["annotated_cds", "candidate", "comparator", "intergenic_lo"]
labels = ["annotated\nCDS", "candidates", "matched\nshadows", "low-3Di\nintergenic"]
w = 0.38
for k, dom in enumerate(DOMS):
    v = [m[(dom, f"prodigal_{a}")][0] * 100 for a in arms]
    xs = np.arange(len(arms)) + (k - 0.5) * w
    ax.bar(xs, v, width=w, color=DOMAIN[dom], label=dom)
    for xx, val in zip(xs, v):
        ax.text(xx, val + 1.5, f"{val:.1f}", ha="center", fontsize=6.2, color=INK)
ax.set_xticks(np.arange(len(arms)), labels)
ax.set_ylabel("% coinciding with a Prodigal call")
ax.set_ylim(0, 108)
ax.legend(frameon=False, loc="upper right")
ax.set_title("by arm (v7 matched comparator)", loc="left")
panel(ax, "a")

ax = axes[1]
order = ["same strand, same frame", "same strand, frameshift",
         "same strand, frame undefined", "opposite strand"]
for k, dom in enumerate(DOMS):
    d = fc[fc.domain == dom].set_index("cds_frame_class").loc[order]
    v = d.n_coincides / d.n * 100
    xs = np.arange(len(order)) + (k - 0.5) * w
    ax.bar(xs, v, width=w, color=DOMAIN[dom], label=dom)
    for xx, val in zip(xs, v):
        ax.text(xx, val + 1.5, f"{val:.1f}", ha="center", fontsize=6.2, color=INK)
ax.set_xticks(np.arange(len(order)), ["same\nframe", "frameshift", "frame\nundefined", "opposite\nstrand"])
ax.set_ylabel("% coinciding")
ax.set_ylim(0, 108)
ax.set_title("all high-3Di shadows, by frame class", loc="left")
panel(ax, "b")

save(fig, "figG05_prodigal")


In [ ]:
for dom in DOMS:
    print(dom, "Prodigal share (clean bg)", "{:.1%} ({:.1%}-{:.1%})".format(*m[(dom, "prodigal_share_clean_bg")]))
print(fc.assign(pct=lambda d: (d.n_coincides / d.n * 100).round(1)).to_string(index=False))


## Caption (draft)

**Figure G05. An independent gene caller agrees with the candidates more often than with their matched shadows.**

**(a)** The percentage of each arm whose ORF coincides (same stop codon and strand) with a Prodigal call. The arms are annotated CDS (positive control), candidates, the matched clean-shadow comparator, and low-3Di intergenic ORFs (unoccupied sequence). Prodigal consults neither GenBank nor any structure database. With the matched comparator as background, the implied Prodigal share is 36.4% (95% CI 36.2–36.6) in bacteria and 42.2% (41.1–43.2) in archaea.

**(b)** Coincidence for the original, unfiltered high-3Di shadow arm, by frame relation to the overlapping annotated CDS (n = 523,346 bacterial and 22,447 archaeal shadows). A same-frame shadow is a sub-ORF of the annotated protein, so Prodigal almost always calls it. That is why only antisense and frameshifted shadows form the comparator.
